# Edge Detection — Fourier, Phase, and Dispersive Optics
Every step symbolic. Operators → Fourier domain → GS phase connection.

In [ ]:
from sympy import *
from IPython.display import display, Math, Markdown

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Gradient: First Derivative Edge Operator

In [ ]:
section('Gradient Operator')
x, omega, sigma, t, z, beta2 = symbols('x omega sigma t z beta_2', real=True)

# 1D signal
f = Function('f')
step('Signal f(x)', f(x))
step('Gradient (1D edge detector): df/dx =', Derivative(f(x), x))

# Gaussian example — derivative = edge location
g = exp(-x**2 / (2*sigma**2))
dg = diff(g, x)
step('Gaussian G(x) =', g)
step('dG/dx =', dg)
step('Simplified =', simplify(dg))
step('Zero crossing at x =', solve(dg, x))
step('→ Edge at x=0 (center of Gaussian)')

## §2 — Fourier Domain: Differentiation = Multiply by iω

In [ ]:
section('Differentiation Theorem')

# The core theorem
F = Function('F')  # Fourier transform of f
step('Fourier transform pair: f(x) ↔ F(ω)')
step('Differentiation theorem:', Eq(
    Derivative(f(x), x),
    Symbol('iω') * F(omega)
))
step('Second derivative:', Eq(
    Derivative(f(x), x, 2),
    -omega**2 * F(omega)
))
step('nth derivative:', Eq(
    Derivative(f(x), x, Symbol('n')),
    (I*omega)**Symbol('n') * F(omega)
))

# Verify with Gaussian
from sympy import fourier_transform
a = symbols('a', positive=True)
gauss = exp(-a*x**2)
F_gauss = fourier_transform(gauss, x, omega)
step('F{e^(-ax²)} =', simplify(F_gauss))
dg_ft = fourier_transform(diff(gauss, x), x, omega)
step('F{d/dx e^(-ax²)} =', simplify(dg_ft))
step('= iω · F{e^(-ax²)} =', simplify(I*omega * F_gauss))

## §3 — Laplacian: Second Derivative, Zero Crossings

In [ ]:
section('Laplacian Operator')

step('1D Laplacian = d²f/dx²')
step('Fourier domain: ∇²f ↔ −ω²F(ω)')

# LoG: Laplacian of Gaussian
G = exp(-x**2 / (2*sigma**2))
dG2 = diff(G, x, 2)
dG2_simplified = simplify(dG2)
step('G(x) = e^(-x²/2σ²) =', G)
step('d²G/dx² =', dG2_simplified)
step('Factor out:', factor(dG2))

# Zero crossings of LoG
zeros = solve(dG2, x)
step('Zero crossings (edge locations): x =', zeros)
step('→ Edges at x = ±σ (inflection points of Gaussian)')

# Fourier of LoG
LoG_ft = fourier_transform(dG2, x, omega)
step('F{LoG} =', simplify(LoG_ft))
step('= −ω² · F{G(ω)} — suppresses DC, amplifies edges')

## §4 — Edge Operators in Fourier Space: Comparison Table

In [ ]:
section('Operator Comparison')

display(Markdown(r'''
| Operator | Spatial domain | Fourier domain | Edge response |
|---|---|---|---|
| Gradient | $\partial f/\partial x$ | $i\omega \cdot F$ | peaks at edges |
| Laplacian | $\partial^2 f/\partial x^2$ | $-\omega^2 \cdot F$ | zero crossings at edges |
| LoG | $\nabla^2 G_\sigma * f$ | $-\omega^2 e^{-\sigma^2\omega^2/2} F$ | zero crossings, smoothed |
| Ideal HP filter | threshold $ |\omega| > \omega_c$ | $F \cdot \mathbf{1}_{|\omega|>\omega_c}$ | all high-freq edges |
| Phase congruency | alignment of harmonics | max $ \sum_n F_n e^{i\phi_n}$ | illumination-invariant edges |
'''))

# Numeric: LoG vs gradient magnitude for a step edge
step('Step edge model: f(x) = Heaviside(x)')
H = Heaviside(x)
step('df/dx = δ(x) — impulse at edge:', diff(H, x))
step('d²f/dx² = δ\'(x) — derivative of delta')

## §5 — Phase and Edges: Why Phase Encodes Structure

In [ ]:
section('Phase Encodes Edges')

display(Markdown(r'''
A signal $f(x)$ has Fourier transform $F(\omega) = |F(\omega)| e^{i\phi(\omega)}$.

**Amplitude** $|F(\omega)|$ — *what* frequencies are present (power spectrum)  
**Phase** $\phi(\omega)$ — *where* those frequencies are aligned (structure, edges)

If you swap phases between two images but keep amplitudes, the reconstructed
image looks like whichever image donated the phase — edges are in the phase.
'''))

# Phase of a shifted delta
x0 = symbols('x_0', real=True)
step('Shifted impulse δ(x−x₀): Fourier transform =', exp(-I*omega*x0))
step('Amplitude |F| =', Abs(exp(-I*omega*x0)))
step('Phase φ(ω) = −ω·x₀  →  linear phase encodes edge position')

# Group delay
phi = -omega * x0
group_delay = -diff(phi, omega)
step('Group delay τ_g = −dφ/dω =', group_delay)
step('→ Group delay = edge position x₀')

## §6 — Dispersive Fourier Transform: Group Delay as Edge Map

In [ ]:
section('Dispersive Fourier Transform (DFT / TDGSA)')

display(Markdown(r'''
In the **Dispersive Fourier Transform** (Jalali lab), a dispersive fiber with
GVD $\beta_2$ maps frequency to time:
$$\tau_g(\omega) = \beta_2 \, z \, \omega$$
Each frequency arrives at a different time — the temporal waveform *is* the
spectrum. This is real-time Fourier analysis at GHz–THz rates.
'''))

# Stationary phase approximation
step('Dispersion phase accumulated over fiber length z:')
phi_disp = Rational(1,2) * beta2 * omega**2 * z
step('φ_disp(ω) = ½ β₂ z ω² =', phi_disp)

tau_g = diff(phi_disp, omega)
step('Group delay τ_g = dφ/dω =', tau_g)
step('= β₂·z·ω  →  time ↔ frequency (linear map)')

# Condition for valid DFT
display(Markdown(r'''
**Far-field (Fraunhofer) condition:**
$$|\beta_2 z| \gg \frac{T_0^2}{2\pi}$$
where $T_0$ is the pulse duration. For $T_0 = 100\,\text{ps}$:
'''))
T0 = 100e-12  # seconds
min_disp = T0**2 / (2*pi)
step(f'|β₂z| ≫ T₀²/2π ≈ {float(min_disp):.3e} ps²/rad')
step('Jalali lab uses |β₂z| ~ 5000–20000 ps² → well into far field')

## §7 — GS Phase Retrieval: Recovering Edges from Intensity

In [ ]:
section('Gerchberg-Saxton Algorithm — Edge Recovery')

display(Markdown(r'''
**Measurements** (what we have):
- $I_t(t) = |\psi(t)|^2$ — time-domain intensity  
- $I_\omega(\omega) = |\Psi(\omega)|^2$ — spectral intensity

**Unknown**: phase $\phi(\omega)$ such that $\Psi = |\Psi|e^{i\phi}$

**GS iteration** (each step is a projection onto a constraint set):

1. Start with random phase guess $\phi^{(0)}(\omega)$
2. **Time constraint**: replace amplitude, keep phase
   $$\psi^{(k)} \leftarrow \sqrt{I_t} \cdot e^{i\angle\psi^{(k)}}$$
3. FFT → frequency domain
4. **Spectral constraint**: replace amplitude, keep phase
   $$\Psi^{(k)} \leftarrow \sqrt{I_\omega} \cdot e^{i\angle\Psi^{(k)}}$$
5. IFFT → back to time domain
6. Repeat until convergence

**Convergence metric**:
$$\epsilon^{(k)} = \frac{\|\,|\psi^{(k)}|^2 - I_t\,\|_2}{\|I_t\|_2}$$
'''))

# TDGSA adds dispersion as second diversity measurement
display(Markdown(r'''
**TDGSA** (two-measurement): adds dispersed intensity $I_d(t)$ from fiber output.
Dispersive constraint replaces spectral constraint:
$$\Psi_d(\omega) = \Psi(\omega) \cdot e^{i\frac{1}{2}\beta_2 z \omega^2}$$
$$\Psi_d \leftarrow \sqrt{I_d(\tau_g^{-1}(t))} \cdot e^{i\angle\Psi_d}$$

Requirement: $|\beta_2 z| \geq 5000\,\text{ps}^2$ for sufficient diversity  
(correlation between $I_t$ and $I_d$ must be $< 0.95$)
'''))

## §8 — Phase Congruency: Illumination-Invariant Edge Detection

In [ ]:
section('Phase Congruency')

display(Markdown(r'''
Classical gradient edges depend on contrast (amplitude). **Phase congruency**
finds edges where Fourier components are maximally *in phase* — independent of
illumination or amplitude.

$$PC(x) = \frac{\sum_n W_n \lfloor A_n \cos(\phi_n(x) - \bar{\phi}(x)) - |\sin(\phi_n(x) - \bar{\phi}(x))| \rfloor}
{\sum_n A_n + \epsilon}$$

where $A_n$, $\phi_n$ are amplitude and phase of the $n$-th Fourier component,
$\bar{\phi}$ is the weighted mean phase.

**Connection to phase retrieval**: if GS recovers $\phi(\omega)$, phase
congruency computed on the recovered field gives edge maps without knowing
absolute intensity calibration.
'''))

# Simple numeric example: sum of harmonics in phase vs out of phase
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

xv = np.linspace(-np.pi, np.pi, 1000)

# In-phase sum → step edge
N = 15
in_phase = sum(np.sin((2*k-1)*xv)/(2*k-1) for k in range(1, N+1))
# Random phase sum → noise
rng = np.random.default_rng(42)
rand_phases = rng.uniform(0, 2*np.pi, N)
random_phase = sum(np.sin((2*k-1)*xv + rand_phases[k-1])/(2*k-1) for k in range(1, N+1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(xv, in_phase)
ax1.set_title('Harmonics in phase → step edge (∑sin((2k-1)x)/(2k-1))')
ax1.set_xlabel('x')
ax2.plot(xv, random_phase, color='orange')
ax2.set_title('Random phases → no coherent edge')
ax2.set_xlabel('x')
plt.tight_layout()
plt.savefig('phase_congruency_demo.png', dpi=100)
plt.show()
print('Edge structure comes entirely from phase alignment, not amplitude.')

## §9 — Polarization and Edge Detection

In [ ]:
section('Polarization')

display(Markdown(r'''
Polarization state of light = 2D complex vector (Jones vector):
$$\mathbf{E} = \begin{pmatrix} E_x \\ E_y \end{pmatrix} = \begin{pmatrix} A_x e^{i\phi_x} \\ A_y e^{i\phi_y} \end{pmatrix}$$

| State | Jones vector |
|---|---|
| Linear H | $(1, 0)^T$ |
| Linear V | $(0, 1)^T$ |
| Linear 45° | $(1, 1)^T/\sqrt{2}$ |
| RCP | $(1, -i)^T/\sqrt{2}$ |
| LCP | $(1, +i)^T/\sqrt{2}$ |
'''))

# Jones calculus
phi_x, phi_y, A_x, A_y = symbols('phi_x phi_y A_x A_y', real=True, positive=True)
Ex = A_x * exp(I*phi_x)
Ey = A_y * exp(I*phi_y)
E = Matrix([Ex, Ey])
step('Jones vector E =', E)
step('Intensity I = E†E =', simplify((Dagger(E).T * E)[0,0]))

# HWP Jones matrix
HWP = Matrix([[1, 0],[0, -1]])
step('Half-wave plate (fast axis H) Jones matrix:', HWP)
step('HWP · E_RCP =', HWP * Matrix([1, -I]) / sqrt(2))
step('→ RCP converts to LCP')

display(Markdown(r'''
**Polarization-based edge detection**: two orthogonal polarization channels
give independent intensity measurements → diversity for phase retrieval
(analogous to dispersive diversity in TDGSA).
'''))

## §10 — Noise, SNR, and Edge Detection Limits

In [ ]:
section('Detection Limits')

N_sig, N_bg = symbols('N_s N_b', positive=True)

# Shot noise SNR
SNR_shot = N_sig / sqrt(N_sig + N_bg)
step('Shot-noise limited SNR =', SNR_shot)
step('High signal limit (N_s ≫ N_b):', limit(SNR_shot, N_bg, 0))
step('= √N_s  →  SNR grows as √(photon count)')

# Rayleigh criterion for edge resolution
lam, NA = symbols('lambda NA', positive=True)
delta_x = Rational(61, 100) * lam / NA  # 0.61 λ/NA
step('Rayleigh resolution limit Δx =', delta_x)
step('At λ=1550 nm, NA=0.1: Δx =', delta_x.subs([(lam, 1550e-9), (NA, 0.1)]), 'meters')

# GS convergence condition
display(Markdown(r'''
**GS convergence** requires measurement diversity. If the two measurements
are too similar (correlation $\rho > 0.95$), the algorithm stagnates.

For TDGSA: correlation decreases as $|\beta_2 z|$ increases.
Empirical minimum: $|\beta_2 z| \geq 5000\,\text{ps}^2$.
'''))